In [74]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

In [75]:
#Reads in the data
df = pd.read_csv('calories_burned_data.csv', na_values='?')

In [76]:
#List out the column names
df.columns

Index(['Gender', 'Age', 'Height(cm)', 'Weight(kg)', 'BMI', 'Running Time(min)',
       'Running Speed(km/h)', 'Distance(km)', 'Average Heart Rate',
       'Calories Burned'],
      dtype='object')

In [77]:
#Showcase the first 5 rows to assess the look of the data. Initial review shows the adata is already in numeric variables
df.head(5)

,Gender,Age,Height(cm),Weight(kg),BMI,Running Time(min),Running Speed(km/h),Distance(km),Average Heart Rate,Calories Burned
0,Male,49,184,74,20.4,78,9.5,3.00,139,973
1,Female,56,182,70,23.2,98,10.2,5.23,165,494
2,Male,49,170,85,27.4,91,10.4,7.61,134,954
3,Male,21,181,59,24.0,89,13.4,14.19,144,570
4,Male,47,172,86,29.8,79,9.8,3.84,149,385


The data takes both Male and Females running statistics and shows how many calories are burned. My target variable will be
the Gender column and my prediction goal is to predict who will burn more calories a Male or Female based on the data presented.

In [78]:
#Look at the data to see if there are any null values. No null values indicated
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Gender               200 non-null    object 
 1   Age                  200 non-null    int64  
 2   Height(cm)           200 non-null    int64  
 3   Weight(kg)           200 non-null    int64  
 4   BMI                  200 non-null    float64
 5   Running Time(min)    200 non-null    int64  
 6   Running Speed(km/h)  200 non-null    float64
 7   Distance(km)         200 non-null    float64
 8   Average Heart Rate   200 non-null    int64  
 9   Calories Burned      200 non-null    int64  
dtypes: float64(3), int64(6), object(1)
memory usage: 15.8+ KB


In [79]:
#Drop any duplicates
df = df.drop_duplicates()

In [80]:
#Rename the Gender column to the target variable
df = df.rename(columns={'Calories Burned': 'Target'})

In [81]:
#Give the Gender variable a numeric label, 0 signifying Males and 1 signifying Females
df['Gender'] = df['Gender'].replace({'Male': 0, 'Female': 1})

C:\Users\local_eharmon4\Temp\ipykernel_9336\1172357064.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['Gender'] = df['Gender'].replace({'Male': 0, 'Female': 1})


In [82]:
# ANOVA test to evaluate fscores and pvalues
from sklearn.feature_selection import SelectKBest, f_classif
x = df.loc[:, ['Gender', 'Age', 'Height(cm)', 'Weight(kg)', 'BMI', 'Running Time(min)', 'Running Speed(km/h)', 
               'Distance(km)', 'Average Heart Rate']]
y = df.loc[:, 'Target']

fs = SelectKBest(score_func=f_classif, k='all')  # call the method
FeaturesS = fs.fit(x, y)    # train the model
np.set_printoptions(suppress = True)
print(FeaturesS.scores_)    # print out the scores 
print()
print(FeaturesS.pvalues_)

[0.74774388 1.47347843 1.25294644 1.11544839 1.30714553 1.10094563
 0.68693459 0.72579394 1.08520105]

[0.83923024 0.15611745 0.28624186 0.40809548 0.24736797 0.42292124
 0.8975623  0.86164149 0.43942614]


In [83]:
#There are no categorical variables that need to be changed to dummy variables due to the features not having more than 2 classes

In [84]:
#Correlation between the data
df.corr()

,Gender,Age,Height(cm),Weight(kg),BMI,Running Time(min),Running Speed(km/h),Distance(km),Average Heart Rate,Target
Gender,1.000000,-0.019329,0.027055,-0.060051,-0.060888,-0.030441,-0.055264,0.061198,-0.137815,-0.095403
Age,-0.019329,1.000000,-0.115622,-0.062736,-0.036499,0.008984,-0.113733,-0.020051,-0.115899,0.035348
Height(cm),0.027055,-0.115622,1.000000,-0.030276,-0.115931,0.112313,-0.028401,0.067200,0.062085,0.100808
Weight(kg),-0.060051,-0.062736,-0.030276,1.000000,-0.050567,-0.109841,0.061732,0.018295,0.017206,0.070512
BMI,-0.060888,-0.036499,-0.115931,-0.050567,1.000000,-0.039732,-0.106841,0.144888,-0.046920,-0.059128
Running Time(min),-0.030441,0.008984,0.112313,-0.109841,-0.039732,1.000000,-0.061102,-0.056769,-0.035617,-0.056699
Running Speed(km/h),-0.055264,-0.113733,-0.028401,0.061732,-0.106841,-0.061102,1.000000,0.008205,-0.010710,0.013207
Distance(km),0.061198,-0.020051,0.067200,0.018295,0.144888,-0.056769,0.008205,1.000000,0.041554,-0.051363
Average Heart Rate,-0.137815,-0.115899,0.062085,0.017206,-0.046920,-0.035617,-0.010710,0.041554,1.000000,0.014701
Target,-0.095403,0.035348,0.100808,0.070512,-0.059128,-0.056699,0.013207,-0.051363,0.014701,1.000000


In [85]:
#Creating a test split dropping certain columns that have a low F score, high pvalue, and low coorelation
#Deciding to drop everything but the Age, Height and BMI columns
x = df.drop(['Gender', 'Weight(kg)', 'Running Time(min)', 'Running Speed(km/h)', 'Distance(km)'], axis=1)
y = df.loc[:, 'Target']
from sklearn.model_selection import train_test_split
x_train, x_test,y_train,y_test = train_test_split(x, y, test_size=0.3,random_state=5)

In [86]:
#No need to balance the data due to appropriate amount of target values
#Build the prediction model using LinearRegression
from sklearn.linear_model import LinearRegression
model = LinearRegression()
model.fit(x_train, y_train)

LinearRegression()

In [53]:
pred = model.predict(x_test)
print('R-squared: %.4f' % model.score(x_test, y_test))  

R-squared: 1.0000


In [98]:
#MAE and R-Squared
#Linear Regression
#Calories Burned as Target